In [ ]:
!pip install transformer-lens

In [ ]:
import torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [ ]:
heads_to_ablate = [
    (5, 5),
    (7, 10),
    (5, 1),
    (6, 9),
    (7, 2),
    (5, 0)
]

def ablate_heads_hook(z, hook):

    layer = hook.layer()

    for (l, h) in heads_to_ablate:
        if l == layer:
            z[:, :, h, :] = 0

    return z

In [ ]:
from functools import partial

all_fwd_hooks = [
    (f"blocks.{layer}.attn.hook_z", ablate_heads_hook)
    for layer in range(model.cfg.n_layers)
]

with model.hooks(
    fwd_hooks=all_fwd_hooks
):
    ablated_logits, ablated_cache = model.run_with_cache(tokens)


In [ ]:
ablated_induction_scores = get_all_induction_scores(
    model,
    ablated_cache,
    prev_pos)

In [ ]:
def single_head_ablation_hook(head_to_ablate):

    def hook_fn(z, hook):

        layer = hook.layer()

        ablate_layer, ablate_head = head_to_ablate

        if layer == ablate_layer:
            z[:, :, ablate_head, :] = 0

        return z

    return hook_fn

In [ ]:
test_heads = [
    (5, 5),
    (7, 10),
    (5, 1),
    (6, 9),
    (7, 2),
    (5, 0)
]


In [ ]:
prompts = [


    "The capital of India is",
    "The largest planet in the solar system is",

    "Once upon a time there was",
    "In a distant galaxy humanity discovered",


    "The reason climate change is difficult to solve is",
    "Artificial intelligence may become dangerous if",


    "Alice went to Paris. Bob went to London. Alice went to",
    "The cat sat on the mat. The dog sat on the",

    "The true meaning of life is",
    "Materialistic Wealth is not sufficient"
]

In [ ]:
def generate_text(
    prompt,
    max_new_tokens=100,
    ablate=False
):

    if not ablate:

        output = model.generate(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=1.0
        )

    else:

        all_fwd_hooks = [
            (
                f"blocks.{layer}.attn.hook_z",
                ablate_heads_hook
            )
            for layer in range(model.cfg.n_layers)
        ]

        with model.hooks(fwd_hooks=all_fwd_hooks):

            output = model.generate(
                prompt,
                max_new_tokens=max_new_tokens,
                temperature=1.0
            )

    return output


In [ ]:
def local_repetition_rate(
    tokens,
    window_size=20
):

    repeated = 0
    total = 0

    for i in range(window_size, len(tokens)):

        current = tokens[i]

        window = tokens[i-window_size:i]

        if current in window:
            repeated += 1

        total += 1

    return repeated / max(total, 1)

In [ ]:
base_scores = []
ablated_scores = []

for prompt in prompts:


    base_text = generate_text(
        prompt,
        ablate=False
    )

    base_tokens = model.to_tokens(base_text)[0].tolist()

    base_lrr = local_repetition_rate(base_tokens)

    base_scores.append(base_lrr)


    ablated_text = generate_text(
        prompt,
        ablate=True
    )

    ablated_tokens = model.to_tokens(ablated_text)[0].tolist()

    ablated_lrr = local_repetition_rate(ablated_tokens)

    ablated_scores.append(ablated_lrr)



    print("=" * 70)
    print("PROMPT:")
    print(prompt)

    print("\nBASE LOCAL REPETITION:")
    print(f"{base_lrr:.4f}")

    print("\nABLATED LOCAL REPETITION:")
    print(f"{ablated_lrr:.4f}")

    print("\nDELTA:")
    print(f"{ablated_lrr - base_lrr:.4f}")


import numpy as np

mean_base = np.mean(base_scores)
mean_ablated = np.mean(ablated_scores)

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(f"Mean Base LRR      : {mean_base:.4f}")
print(f"Mean Ablated LRR   : {mean_ablated:.4f}")
print(f"Mean Delta         : {mean_ablated - mean_base:.4f}")

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

PROMPT:
The capital of India is

BASE LOCAL REPETITION:
0.1163

ABLATED LOCAL REPETITION:
0.4767

DELTA:
0.3605


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

PROMPT:
The true meaning of life is

BASE LOCAL REPETITION:
0.0920

ABLATED LOCAL REPETITION:
0.4368

DELTA:
0.3448


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

PROMPT:
Materialistic Wealth is not sufficient

BASE LOCAL REPETITION:
0.1294

ABLATED LOCAL REPETITION:
0.4598

DELTA:
0.3304


FINAL RESULTS
Mean Base LRR      : 0.1442
Mean Ablated LRR   : 0.4504
Mean Delta         : 0.3062
